In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


- Qwen3-Embedding dùng last-token pooling, nên tokenize bắt buộc phải padding_side = 'left'
- Query cần instruction prefic, passage thì không.

In [2]:
%%capture
!pip install pyvi pyarrow

In [3]:
from pathlib import Path
import sys
import json
import time

import numpy as np
import pandas as pd
import torch

# ---- Model ----
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
RUN_NAME = "Qwen3-Embedding-0.6B"

MODEL_KWARGS = {'torch_dtype': 'float16'}
TOKENIZER_KWARGS = {'padding_side': 'left'}

# Prefix bắt buộc của một số họ model (E5). Để "" nếu model không cần.
QUERY_PREFIX = ""
PASSAGE_PREFIX = ""

# Tách từ tiếng Việt bằng ViTokenizer — bắt buộc với model nền PhoBERT.
USE_WORD_SEGMENTATION = False

# Instruction prompt cho query, sentence-transformers đọc từ config của model (họ Qwen3-Embedding).
QUERY_PROMPT_NAME = 'query'

MAX_PASSAGE_LENGTH = 2048
MAX_QUERY_LENGTH = 64

BATCH_SIZE_PASSAGE = 8
BATCH_SIZE_QUERY = 64

# False = cắt cụt theo giới hạn của model. True = chunk rồi lấy score(doc) = max(score(chunk)).
CHUNKING_MODES = [False]
CHUNK_SIZE = None
CHUNK_OVERLAP = 64

# ---- Đường dẫn ----
BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"

OUTPUT_DIR = RETRIEVER_DIR / "02_Baseline/Dense Retriever/BiEncoder/Output"
EMBEDDING_CACHE_DIR = RETRIEVER_DIR / "02_Baseline/Dense Retriever/BiEncoder/embedding_caches"


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# retriever_common.py đặt cùng thư mục 02_Retriever
sys.path.append(str(RETRIEVER_DIR))

import retriever_common as rc

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
  print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')
query = pd.read_parquet(DATA_DIR / 'corpus' / 'queries.parquet')
qrels = pd.read_parquet(DATA_DIR / 'corpus' / 'qrels.parquet')

assert corpus['doc_id'].is_unique
assert corpus['content_hash'].is_unique, "Corpus còn passage trùng nội dung — chạy lại Preprocessing.ipynb."
assert query['query_id'].is_unique

doc_id_array = corpus['doc_id'].to_numpy()

print(f"Corpus: {len(corpus):,} passage")

for split_name in ['train', 'val', 'test']:
  print(f"{split_name}: {int((query['split'] == split_name).sum()):,} câu hỏi")

Corpus: 294 passage
train: 7,697 câu hỏi
val: 933 câu hỏi
test: 958 câu hỏi


## **02. Chuẩn bị input text**

In [5]:
from pyvi import ViTokenizer

def prepare_passage(text: str) -> str:
  text = str(text)

  if USE_WORD_SEGMENTATION:
    text = ViTokenizer.tokenize(text)

  return PASSAGE_PREFIX + text


def prepare_query(text: str) -> str:
  text = str(text)

  if USE_WORD_SEGMENTATION:
    text = ViTokenizer.tokenize(text)

  return QUERY_PREFIX + text


print("Passage mẫu:\n", prepare_passage(corpus['embedding_text'].iloc[0])[:300], "...\n")
print("Query mẫu:\n", prepare_query(query['query'].iloc[0]))

Passage mẫu:
 Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG
Điều  9.	Tuyển bổ sung và loại ra khỏi chương trình, xét chính thức và dự bị
Đối tượng tham gia CTTN là những sinh viên có năng lực xuất sắc, do đó, sau mỗi học kỳ BĐH quyết định việc loại sinh viên khỏi lớp tài năng, tuyển bổ sung sinh viên từ chương  ...

Query mẫu:
 Sinh viên dự bị không trở thành sinh viên chính thức bao nhiêu học kỳ sẽ bị loại khỏi CTTN?


## **03. Model**

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    MODEL_NAME,
    model_kwargs=MODEL_KWARGS,
    tokenizer_kwargs=TOKENIZER_KWARGS,
    trust_remote_code=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

model.max_seq_length = MAX_PASSAGE_LENGTH

print("Model:", MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("max_seq_length:", model.max_seq_length)
print("padding_side:", model.tokenizer.padding_side)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Model: Qwen/Qwen3-Embedding-0.6B
Embedding dimension: 1024
max_seq_length: 2048
padding_side: left


/tmp/ipykernel_889/4287941934.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


### **3.1. Test**

In [7]:
smoke_texts = [
    prepare_passage(corpus['embedding_text'].iloc[0]),
    prepare_passage(corpus['embedding_text'].iloc[1]),
]

smoke_embeddings = model.encode(smoke_texts, normalize_embeddings=True, convert_to_numpy=True)

print("Shape:", smoke_embeddings.shape)
print("Norm :", np.linalg.norm(smoke_embeddings, axis=1).round(4))

# 1. Chuẩn hoá phải cho norm = 1.
assert np.allclose(np.linalg.norm(smoke_embeddings, axis=1), 1.0, atol=1e-3), "Vector chưa được chuẩn hoá."

# 2. Một passage phải giống chính nó gần như tuyệt đối.
self_similarity = float(smoke_embeddings[0] @ smoke_embeddings[0])
print("Self-similarity:", round(self_similarity, 6))
assert self_similarity > 0.99

# 3. Hai passage khác nhau không được giống hệt nhau. Nếu ~1.0 thì rất có thể pooling sai
#    (ví dụ quên padding_side='left' với model last-token pooling) và mọi vector đang bị đổ về một điểm.
cross_similarity = float(smoke_embeddings[0] @ smoke_embeddings[1])
print("Cross-similarity:", round(cross_similarity, 6))

assert cross_similarity < 0.99, ("Hai passage khác nhau cho vector gần như trùng nhau — kiểm tra lại pooling và padding_side.")

print("\nSmoke test đã pass.")

Shape: (2, 1024)
Norm : [1. 1.]
Self-similarity: 1.0
Cross-similarity: 0.527344

Smoke test đã pass.


## **04. Encode & Retriever**

In [8]:
def encode_texts(texts: list[str], batch_size: int, prompt_name=None) -> np.ndarray:
  encode_kwargs = dict(
      batch_size=batch_size,
      normalize_embeddings=True,
      convert_to_numpy=True,
      show_progress_bar=True,
  )

  if prompt_name is not None:
    encode_kwargs['prompt_name'] = prompt_name

  return model.encode(texts, **encode_kwargs).astype(np.float32)

In [9]:
def encode_corpus(use_chunking: bool):
  # Trả về (embedding_matrix, chunk_table). chunk_table = None khi không chunk
  cache_suffix = 'chunked' if use_chunking else 'truncated'
  cache_path = EMBEDDING_CACHE_DIR / f"{RUN_NAME}__{cache_suffix}.npy"
  chunk_table_path = EMBEDDING_CACHE_DIR / f"{RUN_NAME}__{cache_suffix}_chunks.parquet"

  if use_chunking:
    chunk_table = rc.build_chunk_table(
        corpus.assign(prepared_text=corpus['embedding_text'].map(prepare_passage)),
        tokenizer=model.tokenizer,
        max_tokens=CHUNK_SIZE,
        overlap_tokens=CHUNK_OVERLAP,
        text_column='prepared_text'
    )

    print(f"Chunking: {len(corpus):,} passage -> {len(chunk_table):,} chunk "
          f"({len(chunk_table) / len(corpus):.2f} chunk/passage)")

    texts_to_encode = chunk_table['chunk_text'].tolist()
  else:
    chunk_table = None
    texts_to_encode = [prepare_passage(text) for text in corpus['embedding_text']]

  if cache_path.exists():
    embedding_matrix = np.load(cache_path)

    if embedding_matrix.shape[0] == len(texts_to_encode):
      print(f"Đã nạp embedding từ cache: {cache_path.name} {embedding_matrix.shape}")

      if use_chunking:
        chunk_table = pd.read_parquet(chunk_table_path)

      return embedding_matrix, chunk_table

    print("Cache không khớp số dòng, encode lại.")

  model.max_seq_length = CHUNK_SIZE if use_chunking else MAX_PASSAGE_LENGTH

  start_time = time.perf_counter()

  embedding_matrix = encode_texts(texts_to_encode, batch_size=BATCH_SIZE_PASSAGE)

  print(f"Encode corpus xong trong {time.perf_counter() - start_time:.1f}s -> {embedding_matrix.shape}")

  np.save(cache_path, embedding_matrix)

  if use_chunking:
    chunk_table.to_parquet(chunk_table_path, index=False)

  return embedding_matrix, chunk_table

In [10]:
def run_configuration(use_chunking: bool):
  configuration_name = f"{RUN_NAME}__{'chunked' if use_chunking else 'truncated'}"

  print("=" * 70)
  print(configuration_name)
  print("=" * 70)

  passage_embeddings, chunk_table = encode_corpus(use_chunking)

  ranking_by_split = {}
  per_query_by_split = {}
  summary_records = []

  for split_name in ['val', 'test']:
    split_query = query[query['split'] == split_name]

    model.max_seq_length = MAX_QUERY_LENGTH

    query_embeddings = encode_texts(
        [prepare_query(text) for text in split_query['query']],
        batch_size=BATCH_SIZE_QUERY,
        prompt_name=QUERY_PROMPT_NAME,
    )

    # Vector đã chuẩn hoá -> tích vô hướng = cosine similarity.
    score_matrix = query_embeddings @ passage_embeddings.T

    if use_chunking:
      score_matrix = rc.max_pool_chunk_scores(score_matrix, chunk_table, doc_id_array)

    split_ranking = rc.rankings_from_score_matrix(
        score_matrix, split_query['query_id'].tolist(), doc_id_array, split_name
    )

    split_qrels_map = rc.build_qrels_map(qrels[qrels['split'] == split_name])

    split_per_query = rc.evaluation(split_ranking, split_qrels_map)

    rc.assert_metrics_are_sane(split_per_query)

    ranking_by_split[split_name] = split_ranking
    per_query_by_split[split_name] = split_per_query

    summary = rc.summarize(split_per_query)

    # print(f"[{split_name}] recall@1={summary['recall@1']:.4f}  recall@3={summary['recall@3']:.4f}  "
    #      f"mrr@10={summary['mrr@10']:.4f}  ndcg@10={summary['ndcg@10']:.4f}")


    summary_records.append({
      'split': split_name,
      'number_of_queries': len(split_per_query),
      **rc.summarize(split_per_query).to_dict(),
    })

  summary_dataframe = pd.DataFrame(summary_records)
  summary_dataframe = summary_dataframe.drop(columns = [col for col in summary_dataframe.columns if col.startswith('hit@')])
  display(summary_dataframe.T)

  return configuration_name, ranking_by_split, per_query_by_split, passage_embeddings, chunk_table

In [11]:
results_by_configuration = {}

for use_chunking in CHUNKING_MODES:
  results_by_configuration[use_chunking] = run_configuration(use_chunking)

Qwen3-Embedding-0.6B__truncated
Đã nạp embedding từ cache: Qwen3-Embedding-0.6B__truncated.npy (294, 1024)


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

,0,1
split,val,test
number_of_queries,933,958
recall@1,0.572347,0.564718
mrr@1,0.572347,0.564718
ndcg@1,0.572347,0.564718
recall@3,0.769561,0.776618
mrr@3,0.659521,0.659012
ndcg@3,0.687794,0.689255
recall@5,0.836013,0.836117
mrr@5,0.675223,0.672477


## **05. Best config**

In [12]:
configuration_comparison = []

for use_chunking, (configuration_name, _, per_query_by_split, _, _) in results_by_configuration.items():
  summary = rc.summarize(per_query_by_split['val'])

  configuration_comparison.append({
      'configuration': configuration_name,
      'use_chunking': use_chunking,
      'recall@1': summary['recall@1'],
      'recall@3': summary['recall@3'],
      'recall@10': summary['recall@10'],
      'mrr@10': summary['mrr@10'],
      'ndcg@10': summary['ndcg@10'],
  })

configuration_comparison_df = pd.DataFrame(configuration_comparison).round(4)

display(configuration_comparison_df)

# Chọn theo val, không bao giờ theo test.
BEST_USE_CHUNKING = bool(
    configuration_comparison_df.loc[configuration_comparison_df['mrr@10'].idxmax(), 'use_chunking']
)

print("Cấu hình được chọn: use_chunking =", BEST_USE_CHUNKING)

,configuration,use_chunking,recall@1,recall@3,recall@10,mrr@10,ndcg@10
0,Qwen3-Embedding-0.6B__truncated,False,0.5723,0.7696,0.9068,0.6851,0.739


Cấu hình được chọn: use_chunking = False


## **06. Latency**

In [13]:
best_configuration_name, _, _, best_passage_embeddings, best_chunk_table = results_by_configuration[BEST_USE_CHUNKING]

model.max_seq_length = MAX_QUERY_LENGTH

latency_query_list = query[query['split'] == 'val']['query'].tolist()[:200]


def encode_single_query(query_text: str) -> np.ndarray:
  encode_kwargs = dict(normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

  if QUERY_PROMPT_NAME is not None:
    encode_kwargs['prompt_name'] = QUERY_PROMPT_NAME

  return model.encode([prepare_query(query_text)], **encode_kwargs).astype(np.float32)


def search_single_query(query_vector: np.ndarray):
  score_matrix = query_vector @ best_passage_embeddings.T

  if BEST_USE_CHUNKING:
    score_matrix = rc.max_pool_chunk_scores(score_matrix, best_chunk_table, doc_id_array)

  return np.argsort(-score_matrix[0], kind='stable')[:rc.RANKING_DEPTH]


latency_report = rc.measure_latency(encode_single_query, search_single_query, latency_query_list)

latency_report['peak_vram_gb'] = (
    round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None
)

print(json.dumps(latency_report, ensure_ascii=False, indent=2))

{
  "number_of_queries": 200,
  "encode_ms_per_query": 53.556,
  "search_ms_per_query": 0.134,
  "latency_ms_per_query": 53.69,
  "queries_per_second": 18.6,
  "peak_vram_gb": 1.79
}


## **07. Save results**

In [14]:
_, best_ranking_by_split, best_per_query_by_split, _, _ = results_by_configuration[BEST_USE_CHUNKING]

run_config = {
    'run_name': best_configuration_name,
    'model_name': MODEL_NAME,
    'embedding_dimension': model.get_sentence_embedding_dimension(),
    'use_chunking': BEST_USE_CHUNKING,
    'chunk_size': CHUNK_SIZE if BEST_USE_CHUNKING else None,
    'chunk_overlap': CHUNK_OVERLAP if BEST_USE_CHUNKING else None,
    'max_passage_length': MAX_PASSAGE_LENGTH,
    'max_query_length': MAX_QUERY_LENGTH,
    'query_prefix': QUERY_PREFIX,
    'passage_prefix': PASSAGE_PREFIX,
    'use_word_segmentation': USE_WORD_SEGMENTATION,
    'query_prompt_name': QUERY_PROMPT_NAME,
    'finetuned': False,
    'corpus_size': len(corpus),
    **latency_report,
}

summary_dataframe = rc.save_run(
    OUTPUT_DIR, best_configuration_name,
    best_ranking_by_split, best_per_query_by_split, run_config,
)

display(summary_dataframe)

display(summary_dataframe[['run_name', 'split', 'recall@1', 'recall@3', 'recall@10', 'mrr@10', 'ndcg@10']].round(4))

/tmp/ipykernel_889/3703733943.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  'embedding_dimension': model.get_sentence_embedding_dimension(),


,run_name,split,number_of_queries,recall@1,hit@1,mrr@1,ndcg@1,recall@3,hit@3,mrr@3,...,mrr@20,ndcg@20,recall@25,hit@25,mrr@25,ndcg@25,recall@30,hit@30,mrr@30,ndcg@30
0,Qwen3-Embedding-0.6B__truncated,val,933,0.572347,0.572347,0.572347,0.572347,0.769561,0.769561,0.659521,...,0.687855,0.748594,0.956056,0.956056,0.688380,0.751182,0.961415,0.961415,0.688572,0.752286
1,Qwen3-Embedding-0.6B__truncated,test,958,0.564718,0.564718,0.564718,0.564718,0.776618,0.776618,0.659012,...,0.684720,0.746266,0.957203,0.957203,0.685282,0.749024,0.963466,0.963466,0.685517,0.750332


,run_name,split,recall@1,recall@3,recall@10,mrr@10,ndcg@10
0,Qwen3-Embedding-0.6B__truncated,val,0.5723,0.7696,0.9068,0.6851,0.7390
1,Qwen3-Embedding-0.6B__truncated,test,0.5647,0.7766,0.9071,0.6821,0.7368


### **8.1. Bảng xếp hạng tổng hợp**

In [15]:
leaderboard = rc.collect_leaderboard(OUTPUT_DIR)

display(leaderboard[leaderboard['split'] == 'val'].round(4))

,run_name,split,number_of_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10
5,Qwen3-Embedding-4B__truncated,val,933,0.6463,0.8264,0.8832,0.9346,0.7464,0.7923
6,bge-m3__truncated,val,933,0.6206,0.8124,0.8660,0.9293,0.7269,0.7761
7,multilingual-e5-large__chunked,val,933,0.5884,0.8221,0.8735,0.9453,0.7158,0.7718
8,Qwen3-Embedding-0.6B__truncated,val,933,0.5723,0.7696,0.8360,0.9068,0.6851,0.7390
9,vietnamese-bi-encoder__chunked,val,933,0.5198,0.7224,0.7953,0.8639,0.6371,0.6921


### **8.2. Ranking trên tập train**

In [16]:
EXPORT_TRAIN_RANKING = True

if EXPORT_TRAIN_RANKING:
  train_query = query[query['split'] == 'train']

  model.max_seq_length = MAX_QUERY_LENGTH

  train_query_embeddings = encode_texts(
      [prepare_query(text) for text in train_query['query']],
      batch_size=BATCH_SIZE_QUERY,
      prompt_name=QUERY_PROMPT_NAME,
  )

  train_score_matrix = train_query_embeddings @ best_passage_embeddings.T

  if BEST_USE_CHUNKING:
    train_score_matrix = rc.max_pool_chunk_scores(train_score_matrix, best_chunk_table, doc_id_array)

  train_ranking = rc.rankings_from_score_matrix(
      train_score_matrix, train_query['query_id'].tolist(), doc_id_array, 'train'
  )

  train_positive_pairs = set(
      zip(qrels.loc[qrels['split'] == 'train', 'query_id'],
          qrels.loc[qrels['split'] == 'train', 'doc_id'])
  )

  train_ranking['is_positive'] = [
      (query_id, doc_id) in train_positive_pairs
      for query_id, doc_id in zip(train_ranking['query_id'], train_ranking['doc_id'])
  ]

  train_ranking.to_parquet(f"/content/drive/MyDrive/Legal_Sytem/02_Retriever/02_Baseline/Dense Retriever/BiEncoder/Output/Train_ranking/{best_configuration_name}__train_ranking.parquet", index=False)

  print(f"train_ranking: {len(train_ranking):,} dòng")

Batches:   0%|          | 0/121 [00:00<?, ?it/s]

train_ranking: 384,850 dòng
